# Mode-Drive Explorer

Herhangi bir protein icin ANM Mode-Drive pipeline'i calistir ve sonuclari experimental yapilarla karsilastir.

**Girisler:**
- UniProt ID (experimental yapilari RCSB'den cekmek icin)
- Initial PDB (baslangic yapisi)
- Chain ID

**Ciktilar:**
- 2D TM-RMSD haritasi (experimental + trajectory)
- Step-by-step trajectory (confidence renkli)
- Initial vs final karsilastirma
- Per-step metrik tablolari

## 1. Environment Setup

In [ ]:
import os, sys, shutil

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    if os.path.exists('/content/ANM-openfold3'):
        shutil.rmtree('/content/ANM-openfold3')
    !git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

    if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
        !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
        !pip install -e /content/ANM-openfold3/openfold3-repo -q
    else:
        try:
            import openfold3
        except ImportError:
            !pip install -e /content/ANM-openfold3/openfold3-repo -q

    !pip install -q biopython matplotlib numpy torch pandas requests

    _repo_root = '/content/ANM-openfold3'
else:
    _repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

if IN_COLAB:
    of3_root = '/content/ANM-openfold3/openfold3-repo'
    if of3_root not in sys.path:
        sys.path.insert(0, of3_root)
    os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)
    from openfold3.entry_points.parameters import (
        download_model_parameters, get_default_checkpoint_dir, DEFAULT_CHECKPOINT_NAME,
    )
    _param_dir = get_default_checkpoint_dir()
    _param_dir.mkdir(parents=True, exist_ok=True)
    download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)

print(f'Repo root: {_repo_root}')
print('Environment ready.')

## 2. Kullanici Giris Parametreleri

Asagidaki hucreyi duzenleyin:

In [ ]:
# ════════════════════════════════════════════════════════════════
#  KULLANICI GIRISLERI — Buraya kendi proteinini gir
# ════════════════════════════════════════════════════════════════

UNIPROT_ID   = 'P69441'       # Adenylate Kinase (ornek)
INITIAL_PDB  = '1AKE'         # Baslangic yapisi (apo/holo)
CHAIN_ID     = 'A'            # Zincir
N_STEPS      = 20             # Pipeline step sayisi

# ════════════════════════════════════════════════════════════════
#  CIKTI DIZINI — tum PNG'ler ve CSV buraya kaydedilir
# ════════════════════════════════════════════════════════════════
if IN_COLAB:
    SAVE_DIR = f'/content/drive/MyDrive/ANM-openfold3/explorer/{UNIPROT_ID}_{INITIAL_PDB}'
else:
    SAVE_DIR = os.path.join(_repo_root, 'results', f'explorer_{UNIPROT_ID}_{INITIAL_PDB}')
os.makedirs(SAVE_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════
#  PIPELINE CONFIG — V6 grid search'ten en iyi parametreler
# ════════════════════════════════════════════════════════════════

# Selective mixing (V6 best)
SELECTIVE_MIXING     = True
CHANGE_CUTOFF        = 0.15
ALPHA_BASE           = 0.05
ALPHA_MAX            = 0.7
MAPPING              = 'sigmoid'
W_CONNECTIVITY       = 0.5
W_DISTANCE           = 0.5
DIAGONAL_BAND        = 2

# Pipeline
Z_MIXING_ALPHA       = 0.7
COMBINATION_STRATEGY = 'autostop'
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = 'plus'

# Composite gate
USE_COMPOSITE_GATE       = True
COMPOSITE_GATE_THRESHOLD = 0.55
GATE_W_CR                = 0.45
GATE_W_PLDDT             = 0.30
GATE_W_RG                = 0.15
GATE_W_PTM               = 0.10

# Hard cutoffs
CONFIDENCE_PTM_CUTOFF     = 0.15
CONFIDENCE_PLDDT_CUTOFF   = 65.0
CONFIDENCE_RG_MAX         = 2.5
CONFIDENCE_RG_MIN         = 0.3
CONFIDENCE_CLASH_REJECT   = True
CONFIDENCE_RMSD_INIT_MAX  = 10.0

# Stall & drift
ALPHA_DECAY           = 0.8
MAX_STALL             = 8
ENABLE_BEST_ROLLBACK  = True
BEST_ROLLBACK_TM_DROP = 0.40
ENABLE_ADAPTIVE_STOP  = True
ADAPTIVE_STOP_WINDOW  = 3
MAX_ROLLBACK_STOP     = 5   # N rollback sonrasi dur

# Fallback
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True
AUTOSTOP_FALLBACK_LEVELS  = (0, 1, 4, 9)

# Autostop
AUTOSTOP_V_MAGNITUDE = 1.0
AUTOSTOP_N_STEPS     = 5000
AUTOSTOP_BACK_OFF    = 2

# OF3
USE_MSA_SERVER        = True
USE_TEMPLATES         = False
NUM_ROLLOUT_STEPS     = None
NUM_DIFFUSION_SAMPLES = 1

# Converter
DRIVE_DIR = '/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa'
CHECKPOINT_SELECTION = 'best_bf_r'

print(f'Protein: UniProt={UNIPROT_ID}, Initial PDB={INITIAL_PDB}, Chain={CHAIN_ID}')
print(f'Pipeline: {N_STEPS} steps, selective={SELECTIVE_MIXING}')
print(f'Selective: cutoff={CHANGE_CUTOFF} base={ALPHA_BASE} max={ALPHA_MAX} map={MAPPING} band={DIAGONAL_BAND}')
print(f'Save dir: {SAVE_DIR}')

## 3. RCSB'den Experimental Yapilari Cek

In [ ]:
import requests
import json as _json
import torch
import numpy as np
from Bio.PDB import PDBParser, PDBList
from Bio.SeqUtils import seq1


def fetch_ca(pdb_id: str, chain_id: str):
    """PDB'den CA koordinatlarini ve sekansini cek."""
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb_cache', file_format='pdb')
    structure = parser.get_structure(pdb_id, pdb_file)
    chain = [c for c in structure[0].get_chains() if c.id == chain_id][0]
    residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    coords = torch.tensor([r['CA'].get_vector().get_array() for r in residues], dtype=torch.float32)
    sequence = ''.join(seq1(r.get_resname()) for r in residues)
    return coords, sequence


def fetch_pdb_ids_for_uniprot(uniprot_id: str) -> list[dict]:
    """RCSB Search API ile UniProt ID'ye ait tum PDB yapilarini bul."""
    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                "operator": "exact_match",
                "value": uniprot_id
            }
        },
        "request_options": {
            "return_all_hits": True
        },
        "return_type": "entry"
    }
    resp = requests.post(
        'https://search.rcsb.org/rcsbsearch/v2/query',
        json=query, timeout=30,
    )
    if resp.status_code != 200:
        print(f'RCSB search failed: {resp.status_code}')
        return []
    data = resp.json()
    results = data.get('result_set', [])
    return [{'pdb_id': r['identifier']} for r in results]


def get_pdb_info(pdb_id: str) -> dict:
    """PDB'nin resolution ve method bilgisini cek."""
    url = f'https://data.rcsb.org/rest/v1/core/entry/{pdb_id}'
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return {'method': '?', 'resolution': None}
        d = resp.json()
        method = d.get('exptl', [{}])[0].get('method', '?')
        res_data = d.get('rcsb_entry_info', {})
        resolution = res_data.get('resolution_combined', [None])
        res_val = resolution[0] if resolution else None
        return {'method': method, 'resolution': res_val}
    except Exception:
        return {'method': '?', 'resolution': None}


# ── Initial yapi ──
print(f'Initial yapi yukleniyor: {INITIAL_PDB} chain {CHAIN_ID}...')
initial_ca, sequence = fetch_ca(INITIAL_PDB, CHAIN_ID)
N = initial_ca.shape[0]
print(f'  N={N} residue, seq len={len(sequence)}')

# ── Experimental yapilar ──
print(f'\nRCSB\'den {UNIPROT_ID} icin yapilar aranıyor...')
pdb_entries = fetch_pdb_ids_for_uniprot(UNIPROT_ID)
print(f'  {len(pdb_entries)} PDB bulundu')

# Her yapiyi yukle ve CA koordinatlarini cek
from src.mode_drive_utils import compute_rmsd, tm_score

experimental_structures = []
for entry in pdb_entries:
    pdb_id = entry['pdb_id']
    try:
        ca, seq = fetch_ca(pdb_id, CHAIN_ID)
        # Sadece ayni uzunluktaki yapilari al
        if ca.shape[0] != N:
            print(f'  {pdb_id}: N={ca.shape[0]} != {N}, atlandi')
            continue
        info = get_pdb_info(pdb_id)
        rmsd_to_init = compute_rmsd(ca, initial_ca)
        tm_to_init = tm_score(ca, initial_ca)
        experimental_structures.append({
            'pdb_id': pdb_id,
            'ca': ca,
            'sequence': seq,
            'rmsd_to_init': rmsd_to_init,
            'tm_to_init': tm_to_init,
            'method': info['method'],
            'resolution': info['resolution'],
        })
        print(f'  {pdb_id}: RMSD={rmsd_to_init:.2f}A TM={tm_to_init:.4f} ({info["method"]}, {info["resolution"]}A)')
    except Exception as e:
        print(f'  {pdb_id}: HATA — {e}')

print(f'\nToplam {len(experimental_structures)} experimental yapi yuklendi.')

## 4. Experimental Yapilar Arasi TM-RMSD Matrisi

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

n_exp = len(experimental_structures)
tm_matrix = np.zeros((n_exp, n_exp))
rmsd_matrix = np.zeros((n_exp, n_exp))
pdb_labels = [s['pdb_id'] for s in experimental_structures]

for i in range(n_exp):
    for j in range(n_exp):
        if i == j:
            tm_matrix[i, j] = 1.0
            rmsd_matrix[i, j] = 0.0
        elif j > i:
            ca_i = experimental_structures[i]['ca']
            ca_j = experimental_structures[j]['ca']
            tm_val = tm_score(ca_i, ca_j)
            rmsd_val = compute_rmsd(ca_i, ca_j)
            tm_matrix[i, j] = tm_val
            tm_matrix[j, i] = tm_val
            rmsd_matrix[i, j] = rmsd_val
            rmsd_matrix[j, i] = rmsd_val

# Heatmap: TM-score matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(tm_matrix, cmap='RdYlGn', vmin=0.3, vmax=1.0, aspect='equal')
axes[0].set_xticks(range(n_exp))
axes[0].set_xticklabels(pdb_labels, rotation=45, ha='right', fontsize=7)
axes[0].set_yticks(range(n_exp))
axes[0].set_yticklabels(pdb_labels, fontsize=7)
axes[0].set_title('TM-score (experimental arasi)')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(rmsd_matrix, cmap='RdYlGn_r', vmin=0, vmax=10, aspect='equal')
axes[1].set_xticks(range(n_exp))
axes[1].set_xticklabels(pdb_labels, rotation=45, ha='right', fontsize=7)
axes[1].set_yticks(range(n_exp))
axes[1].set_yticklabels(pdb_labels, fontsize=7)
axes[1].set_title('RMSD (experimental arasi)')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.suptitle(f'{UNIPROT_ID} — {n_exp} Experimental Yapi', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, '01_experimental_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'  Saved: 01_experimental_matrix.png')

# Tablo
rows = []
for s in experimental_structures:
    rows.append({
        'PDB': s['pdb_id'],
        'RMSD_to_init': f"{s['rmsd_to_init']:.2f}",
        'TM_to_init': f"{s['tm_to_init']:.4f}",
        'Method': s['method'][:15],
        'Res(A)': f"{s['resolution']:.1f}" if s['resolution'] else '?',
    })
df_exp = pd.DataFrame(rows)
print(df_exp.to_string(index=False))
df_exp.to_csv(os.path.join(SAVE_DIR, '01_experimental_structures.csv'), index=False)

## 5. Converter + OF3 Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import time

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score

# Converter
history_path = os.path.join(DRIVE_DIR, 'history.json')
best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
    with open(history_path) as f:
        history = _json.load(f)
    best_bf_r = -1
    best_epoch = -1
    for entry in history:
        bf_r = entry.get('val_bf_pearson', 0)
        if bf_r > best_bf_r:
            best_bf_r = bf_r
            best_epoch = entry['epoch']
    epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
    CHECKPOINT_PATH = epoch_ckpt if os.path.exists(epoch_ckpt) else best_model_path
    print(f'Best bf_r: {best_bf_r:.4f} at epoch {best_epoch}')
else:
    CHECKPOINT_PATH = best_model_path

converter = PairContactConverter(CHECKPOINT_PATH)
print(f'Converter loaded.')

# OF3
_query_dir = '/content/of3_queries'
os.makedirs(_query_dir, exist_ok=True)
_query = {
    'queries': {
        INITIAL_PDB: {
            'chains': [{'molecule_type': 'protein', 'chain_ids': [CHAIN_ID], 'sequence': sequence}]
        }
    }
}
_query_path = os.path.join(_query_dir, f'{INITIAL_PDB}.json')
with open(_query_path, 'w') as f:
    _json.dump(_query, f, indent=2)

from src.of3_diffusion import load_of3_diffusion
diffusion_fn, zij_trunk = load_of3_diffusion(
    query_json=_query_path,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
    num_rollout_steps=NUM_ROLLOUT_STEPS,
    num_samples=NUM_DIFFUSION_SAMPLES,
)
print(f'zij_trunk shape: {zij_trunk.shape}')

from src.autostop_adapter import StructureContext
_pdb_file = PDBList().retrieve_pdb_file(INITIAL_PDB, pdir='/tmp/pdb_cache', file_format='pdb')
structure_ctx = StructureContext.from_pdb(_pdb_file, chain_id=CHAIN_ID)
print('OF3 ready.')

## 6. Mode-Drive Pipeline Calistir

In [ ]:
# ════════════════════════════════════════════════════════════════
#  PIPELINE CALISTIR
# ════════════════════════════════════════════════════════════════

cfg = ModeDriveConfig(
    n_steps=N_STEPS,
    combination_strategy=COMBINATION_STRATEGY,
    z_mixing_alpha=Z_MIXING_ALPHA,
    n_anm_modes=N_ANM_MODES,
    n_combinations=N_COMBINATIONS,
    max_combo_size=MAX_COMBO_SIZE,
    df=DF, df_min=DF_MIN, df_max=DF_MAX,
    anm_cutoff=ANM_CUTOFF,
    contact_r_cut=CONTACT_R_CUT,
    contact_tau=CONTACT_TAU,
    z_direction=Z_DIRECTION,
    num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
    use_composite_gate=USE_COMPOSITE_GATE,
    composite_gate_threshold=COMPOSITE_GATE_THRESHOLD,
    gate_w_cr=GATE_W_CR,
    gate_w_plddt=GATE_W_PLDDT,
    gate_w_rg=GATE_W_RG,
    gate_w_ptm=GATE_W_PTM,
    confidence_ptm_cutoff=CONFIDENCE_PTM_CUTOFF,
    confidence_plddt_cutoff=CONFIDENCE_PLDDT_CUTOFF,
    confidence_rg_max=CONFIDENCE_RG_MAX,
    confidence_rg_min=CONFIDENCE_RG_MIN,
    confidence_clash_reject=CONFIDENCE_CLASH_REJECT,
    confidence_rmsd_init_max=CONFIDENCE_RMSD_INIT_MAX,
    enable_confidence_fallback=ENABLE_FALLBACK,
    fallback_extended_enabled=FALLBACK_EXTENDED_ENABLED,
    autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
    autostop_n_steps=AUTOSTOP_N_STEPS,
    autostop_back_off=AUTOSTOP_BACK_OFF,
    autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
    autostop_chain_id=CHAIN_ID,
    rejected_alpha_decay=ALPHA_DECAY,
    max_consecutive_rejected=MAX_STALL,
    confidence_warmup_steps=0,
    enable_best_rollback=ENABLE_BEST_ROLLBACK,
    best_rollback_tm_drop=BEST_ROLLBACK_TM_DROP,
    enable_adaptive_stop=ENABLE_ADAPTIVE_STOP,
    adaptive_stop_window=ADAPTIVE_STOP_WINDOW,
    # Selective mixing
    selective_mixing=SELECTIVE_MIXING,
    selective_w_connectivity=W_CONNECTIVITY,
    selective_w_distance=W_DISTANCE,
    selective_change_cutoff=CHANGE_CUTOFF,
    selective_alpha_base=ALPHA_BASE,
    selective_alpha_max=ALPHA_MAX,
    selective_mapping=MAPPING,
    selective_diagonal_band=DIAGONAL_BAND,
)

pipe = ModeDrivePipeline(
    converter=converter, config=cfg,
    diffusion_fn=diffusion_fn,
    structure_ctx=structure_ctx,
)

# ── Run loop ──
coords_ca = initial_ca.clone()
z_current = zij_trunk.clone()
orig_alpha = Z_MIXING_ALPHA
current_alpha = orig_alpha
consecutive_rejected = 0

coords_best = initial_ca.clone()
z_best = zij_trunk.clone()
rollback_count = 0

# Trajectory: her accepted step'in CA koordinatlarini sakla
trajectory_coords = [initial_ca.clone()]  # step 0 = initial
step_metrics = []

t0 = time.time()

for step_idx in range(N_STEPS):
    cfg.z_mixing_alpha = current_alpha
    sr = pipe.step_with_fallback(
        coords_ca, initial_ca, z_current,
        prev_rmsd=0.0,
        target_coords=None,  # hedef bilinmiyor
        step_idx=step_idx,
    )

    accepted = not sr.rejected
    reject_reason = ''
    if sr.rg_ratio is not None and sr.rg_ratio > CONFIDENCE_RG_MAX:
        accepted = False
        reject_reason = f'Rg={sr.rg_ratio:.1f}>{CONFIDENCE_RG_MAX}'
    if sr.has_clash and CONFIDENCE_CLASH_REJECT:
        accepted = False
        reject_reason = 'clash'
    if sr.rmsd > CONFIDENCE_RMSD_INIT_MAX:
        accepted = False
        reject_reason = f'rmsd_init={sr.rmsd:.1f}>{CONFIDENCE_RMSD_INIT_MAX}'

    # Her experimental yapiya karsi TM ve RMSD hesapla
    exp_scores = {}
    for s in experimental_structures:
        exp_tm = tm_score(sr.new_ca, s['ca'])
        exp_rmsd = compute_rmsd(sr.new_ca, s['ca'])
        exp_scores[s['pdb_id']] = {'tm': exp_tm, 'rmsd': exp_rmsd}

    # En yakin experimental yapi
    best_exp = max(exp_scores.items(), key=lambda x: x[1]['tm'])

    plddt_mean = float(sr.plddt.mean()) if sr.plddt is not None else None
    m = {
        'step': step_idx + 1,
        'accepted': accepted,
        'rmsd_init': sr.rmsd,
        'ptm': sr.ptm,
        'plddt_mean': plddt_mean,
        'ranking': sr.ranking_score,
        'mean_pae': sr.mean_pae,
        'rg_ratio': sr.rg_ratio,
        'contact_recon': sr.contact_recon,
        'has_clash': sr.has_clash,
        'fallback_level': sr.fallback_level,
        'alpha_used': current_alpha,
        'reject_reason': reject_reason,
        'rollback': False,
        'exp_scores': exp_scores,
        'best_exp_pdb': best_exp[0],
        'best_exp_tm': best_exp[1]['tm'],
        'best_exp_rmsd': best_exp[1]['rmsd'],
    }

    tag = 'ACC' if accepted else 'REJ'
    plddt_str = f"{plddt_mean:.0f}" if plddt_mean else '?'
    cr_str = f"{sr.contact_recon:.2f}" if sr.contact_recon is not None else '?'
    extra = f' {reject_reason}' if not accepted else ''
    print(
        f'  [{step_idx+1:>2}/{N_STEPS}] {tag}  '
        f'pTM={sr.ptm:.3f} pLDDT={plddt_str} cR={cr_str} '
        f'RMSD_init={sr.rmsd:.2f} '
        f'best_exp={best_exp[0]}(TM={best_exp[1]["tm"]:.3f}) '
        f'FL={sr.fallback_level}{extra}'
    )

    if accepted:
        trajectory_coords.append(sr.new_ca.clone())
        coords_ca = sr.new_ca
        z_current = sr.z_modified
        consecutive_rejected = 0
        current_alpha = orig_alpha

        # Rollback check: en iyi experimental TM ile karsilastir
        if best_exp[1]['tm'] > getattr(pipe, '_best_exp_tm', 0):
            pipe._best_exp_tm = best_exp[1]['tm']
            coords_best = sr.new_ca.clone()
            z_best = sr.z_modified.clone()
        elif pipe._best_exp_tm > 0 and best_exp[1]['tm'] < pipe._best_exp_tm * (1.0 - BEST_ROLLBACK_TM_DROP):
            print(f'  >> ROLLBACK (best_exp_tm drop)')
            coords_ca = coords_best.clone()
            z_current = z_best.clone()
            rollback_count += 1
            m['rollback'] = True
            if rollback_count >= MAX_ROLLBACK_STOP:
                print(f'  >> STOP: {rollback_count} rollback')
                step_metrics.append(m)
                break
    else:
        consecutive_rejected += 1
        if ALPHA_DECAY < 1.0:
            current_alpha = max(0.02, current_alpha * ALPHA_DECAY)
        if MAX_STALL > 0 and consecutive_rejected >= MAX_STALL:
            print(f'  STALL: {consecutive_rejected} consecutive rejected')
            step_metrics.append(m)
            break

    step_metrics.append(m)

wall = time.time() - t0
n_accepted = sum(1 for m in step_metrics if m['accepted'])
print(f'\nTamamlandi: {n_accepted}/{len(step_metrics)} accepted, {wall:.0f}s')
print(f'Trajectory: {len(trajectory_coords)} frame (initial + {len(trajectory_coords)-1} accepted step)')

## 7. 2D TM-RMSD Landscape + Trajectory

In [ ]:
# ════════════════════════════════════════════════════════════════
#  2D TM-RMSD MAP: Experimental yapilar + trajectory
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(14, 8))

# -- Experimental yapilar (yildiz) --
for s in experimental_structures:
    x = s['rmsd_to_init']
    y = s['tm_to_init']
    is_initial = (s['pdb_id'].upper() == INITIAL_PDB.upper())
    color = '#e74c3c' if is_initial else '#2ecc71'
    size = 300 if is_initial else 180
    ax.scatter(x, y, marker='*', s=size, c=color, edgecolors='black',
              linewidths=0.8, zorder=10)
    ax.annotate(s['pdb_id'], (x, y), textcoords='offset points',
               xytext=(5, 5), fontsize=7, alpha=0.8)

# -- Trajectory (accepted steps) --
traj_x = []
traj_y = []
traj_c = []
traj_labels = []

traj_x.append(0.0)
traj_y.append(1.0)
traj_c.append(90.0)
traj_labels.append('init')

for m in step_metrics:
    if not m['accepted']:
        continue
    traj_x.append(m['rmsd_init'])
    traj_y.append(m['best_exp_tm'])
    traj_c.append(m['plddt_mean'] if m['plddt_mean'] is not None else 70.0)
    traj_labels.append(f"S{m['step']}")

# Trajectory cizgi
ax.plot(traj_x, traj_y, '-', color='#3498db', alpha=0.4, linewidth=1.5, zorder=5)

# Trajectory noktalar (confidence renkli)
sc = ax.scatter(traj_x, traj_y, c=traj_c, cmap='RdYlGn', vmin=60, vmax=90,
               s=100, edgecolors='black', linewidths=0.5, zorder=8, marker='o')
plt.colorbar(sc, ax=ax, label='pLDDT', shrink=0.7)

for i, label in enumerate(traj_labels):
    ax.annotate(label, (traj_x[i], traj_y[i]), textcoords='offset points',
               xytext=(5, -10), fontsize=6, color='#2c3e50', alpha=0.8)

ax.set_xlabel('RMSD to Initial (A)', fontsize=12)
ax.set_ylabel('Best TM to Experimental', fontsize=12)
ax.set_title(f'{UNIPROT_ID} — Conformational Landscape\n'
             f'Stars=Experimental, Circles=Mode-Drive Trajectory (color=pLDDT)', fontsize=13)
ax.grid(True, alpha=0.2)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, '02_landscape_trajectory.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'  Saved: 02_landscape_trajectory.png')

## 8. Trajectory vs Her Experimental Yapi

In [ ]:
# ════════════════════════════════════════════════════════════════
#  Her experimental yapiya karsi TM trajectory
# ════════════════════════════════════════════════════════════════

accepted_steps = [m for m in step_metrics if m['accepted']]

if experimental_structures and accepted_steps:
    non_initial = [s for s in experimental_structures if s['pdb_id'].upper() != INITIAL_PDB.upper()]
    non_initial.sort(key=lambda s: s['tm_to_init'])
    show_targets = non_initial[:8]

    fig, ax = plt.subplots(figsize=(14, 5))
    steps = [m['step'] for m in accepted_steps]
    colors_cycle = plt.cm.tab10(np.linspace(0, 1, len(show_targets)))

    for idx, target in enumerate(show_targets):
        pdb_id = target['pdb_id']
        tms = [m['exp_scores'][pdb_id]['tm'] for m in accepted_steps]
        ax.plot(steps, tms, 'o-', color=colors_cycle[idx], label=pdb_id,
               markersize=5, linewidth=1.5, alpha=0.8)

    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3, label='TM=0.5 (same fold)')
    ax.set_xlabel('Step')
    ax.set_ylabel('TM-score')
    ax.set_title('Trajectory TM to Experimental Structures')
    ax.legend(fontsize=7, ncol=2, loc='best')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, '03_trajectory_vs_experimental.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: 03_trajectory_vs_experimental.png')
else:
    print('Gosterecek veri yok.')

## 9. Per-Step Confidence Metrikleri

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CONFIDENCE METRIKLERI: pTM, pLDDT, cR, Rg, mPAE
# ════════════════════════════════════════════════════════════════

if accepted_steps:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))

    all_steps = [m['step'] for m in step_metrics]
    acc_mask = [m['accepted'] for m in step_metrics]

    def plot_metric(ax, key, ylabel, title, ylim=None):
        vals = [m.get(key) for m in step_metrics]
        for i, (s, v, a) in enumerate(zip(all_steps, vals, acc_mask)):
            if v is None:
                continue
            color = '#2ecc71' if a else '#e74c3c'
            marker = 'o' if a else 'x'
            ax.scatter(s, v, c=color, marker=marker, s=50, zorder=5, edgecolors='black', linewidths=0.3)
        acc_s = [m['step'] for m in step_metrics if m['accepted'] and m.get(key) is not None]
        acc_v = [m[key] for m in step_metrics if m['accepted'] and m.get(key) is not None]
        if acc_s:
            ax.plot(acc_s, acc_v, '-', color='#3498db', alpha=0.4)
        ax.set_xlabel('Step', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(title, fontsize=10)
        if ylim:
            ax.set_ylim(ylim)
        ax.grid(True, alpha=0.2)

    plot_metric(axes[0,0], 'ptm', 'pTM', 'pTM per Step', ylim=(0, 1))
    plot_metric(axes[0,1], 'plddt_mean', 'pLDDT', 'Mean pLDDT per Step', ylim=(50, 95))
    plot_metric(axes[0,2], 'contact_recon', 'cR', 'Contact Reconstruction', ylim=(0.8, 1.0))
    plot_metric(axes[1,0], 'rg_ratio', 'Rg ratio', 'Radius of Gyration Ratio')
    plot_metric(axes[1,1], 'mean_pae', 'mPAE', 'Mean PAE')
    plot_metric(axes[1,2], 'best_exp_tm', 'TM', 'Best Exp TM per Step')

    plt.suptitle(f'{UNIPROT_ID} — Confidence Metrics (green=accepted, red=rejected)', fontsize=12)
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, '04_confidence_metrics.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: 04_confidence_metrics.png')

## 10. Initial vs Final Karsilastirma

In [ ]:
# ════════════════════════════════════════════════════════════════
#  INITIAL vs BEST vs FINAL karsilastirma
# ════════════════════════════════════════════════════════════════

if accepted_steps:
    best_step = max(accepted_steps, key=lambda m: m['best_exp_tm'])
    best_step_idx = accepted_steps.index(best_step)
    best_coords = trajectory_coords[best_step_idx + 1]
    final_coords = trajectory_coords[-1]
else:
    best_coords = initial_ca
    final_coords = initial_ca
    best_step = {'step': 0, 'best_exp_tm': 0, 'best_exp_rmsd': 0}

comparison_rows = []
for s in experimental_structures:
    pdb_id = s['pdb_id']
    ca = s['ca']
    init_tm = tm_score(initial_ca, ca)
    init_rmsd = compute_rmsd(initial_ca, ca)
    best_tm = tm_score(best_coords, ca)
    best_rmsd = compute_rmsd(best_coords, ca)
    final_tm = tm_score(final_coords, ca)
    final_rmsd = compute_rmsd(final_coords, ca)
    comparison_rows.append({
        'Target PDB': pdb_id,
        'Init TM': f'{init_tm:.4f}',
        'Init RMSD': f'{init_rmsd:.2f}',
        'Best TM': f'{best_tm:.4f}',
        'Best RMSD': f'{best_rmsd:.2f}',
        'Final TM': f'{final_tm:.4f}',
        'Final RMSD': f'{final_rmsd:.2f}',
        'Delta TM': f'{best_tm - init_tm:+.4f}',
    })

df_comp = pd.DataFrame(comparison_rows)
print('='*100)
print(f'INITIAL vs BEST (step {best_step["step"]}) vs FINAL')
print('='*100)
print(df_comp.to_string(index=False))
df_comp.to_csv(os.path.join(SAVE_DIR, '05_initial_vs_best_vs_final.csv'), index=False)

# Bar chart
if len(comparison_rows) > 1:
    fig, ax = plt.subplots(figsize=(12, max(5, len(comparison_rows)*0.4)))
    targets = [r['Target PDB'] for r in comparison_rows]
    init_tms = [float(r['Init TM']) for r in comparison_rows]
    best_tms = [float(r['Best TM']) for r in comparison_rows]

    y_pos = np.arange(len(targets))
    width = 0.35

    ax.barh(y_pos - width/2, init_tms, width, label='Initial', color='#e74c3c', alpha=0.7)
    ax.barh(y_pos + width/2, best_tms, width, label='Best Step', color='#2ecc71', alpha=0.7)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(targets, fontsize=8)
    ax.set_xlabel('TM-score')
    ax.set_title(f'Initial vs Best: TM to Each Experimental Structure')
    ax.legend()
    ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.2, axis='x')
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, '05_initial_vs_best_bar.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: 05_initial_vs_best_bar.png')

## 11. Pairwise TM Heatmap: Trajectory x Experimental

In [ ]:
# ════════════════════════════════════════════════════════════════
#  HEATMAP: Trajectory frames x Experimental structures
# ════════════════════════════════════════════════════════════════

n_frames = len(trajectory_coords)
n_targets = len(experimental_structures)

if n_targets > 0 and n_frames > 1:
    tm_heatmap = np.zeros((n_frames, n_targets))

    frame_labels = ['Initial'] + [f'S{m["step"]}' for m in step_metrics if m['accepted']]
    frame_labels = frame_labels[:n_frames]
    target_labels = [s['pdb_id'] for s in experimental_structures]

    for i in range(n_frames):
        for j in range(n_targets):
            tm_heatmap[i, j] = tm_score(trajectory_coords[i], experimental_structures[j]['ca'])

    fig, ax = plt.subplots(figsize=(max(8, n_targets*0.8), max(6, n_frames*0.35)))
    im = ax.imshow(tm_heatmap, cmap='RdYlGn', vmin=0.3, vmax=1.0, aspect='auto')

    ax.set_xticks(range(n_targets))
    ax.set_xticklabels(target_labels, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(n_frames))
    ax.set_yticklabels(frame_labels, fontsize=7)
    ax.set_xlabel('Experimental Structure')
    ax.set_ylabel('Trajectory Frame')
    ax.set_title(f'{UNIPROT_ID} — TM Heatmap: Trajectory x Experimental')
    plt.colorbar(im, ax=ax, label='TM-score', shrink=0.8)

    for i in range(n_frames):
        for j in range(n_targets):
            val = tm_heatmap[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=5, color=color)

    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, '06_trajectory_heatmap.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: 06_trajectory_heatmap.png')

    print('\nEn iyi trajectory frame per experimental:')
    for j in range(n_targets):
        best_i = np.argmax(tm_heatmap[:, j])
        print(f'  {target_labels[j]:6s} -> {frame_labels[best_i]:8s} TM={tm_heatmap[best_i, j]:.4f}')

## 12. Ozet

In [ ]:
# ════════════════════════════════════════════════════════════════
#  FINAL OZET + KAYIT
# ════════════════════════════════════════════════════════════════

# Step metrics CSV (exp_scores disinda)
metrics_rows = []
for m in step_metrics:
    row = {k: v for k, v in m.items() if k != 'exp_scores'}
    metrics_rows.append(row)
df_metrics = pd.DataFrame(metrics_rows)
df_metrics.to_csv(os.path.join(SAVE_DIR, '07_step_metrics.csv'), index=False)

print('='*80)
print(f'  MODE-DRIVE EXPLORER OZET')
print(f'  Protein: {UNIPROT_ID} / Initial: {INITIAL_PDB} chain {CHAIN_ID}')
print('='*80)
print(f'\n  Experimental yapilar: {len(experimental_structures)}')
print(f'  Pipeline steps: {len(step_metrics)} ({n_accepted} accepted)')
print(f'  Trajectory frames: {len(trajectory_coords)}')
print(f'  Rollbacks: {rollback_count}')
print(f'  Wall time: {wall:.0f}s')

if accepted_steps:
    print(f'\n  En iyi step: {best_step["step"]}')
    print(f'    Best exp match: {best_step["best_exp_pdb"]} TM={best_step["best_exp_tm"]:.4f}')
    print(f'    pTM={best_step["ptm"]:.3f} pLDDT={best_step["plddt_mean"]:.0f}')

    print(f'\n  Experimental yapilar - yaklasim ozeti:')
    for s in experimental_structures:
        pdb_id = s['pdb_id']
        init_tm = tm_score(initial_ca, s['ca'])
        traj_tms = [tm_score(frame, s['ca']) for frame in trajectory_coords]
        max_tm = max(traj_tms)
        max_frame = traj_tms.index(max_tm)
        delta = max_tm - init_tm
        arrow = '+' if delta > 0.01 else ('~' if abs(delta) <= 0.01 else '-')
        print(f'    {pdb_id}: init_TM={init_tm:.3f} -> best_TM={max_tm:.3f} ({arrow}{abs(delta):.3f}) frame={max_frame}')

print(f'\n  Selective mixing config:')
print(f'    cutoff={CHANGE_CUTOFF} base={ALPHA_BASE} max={ALPHA_MAX}')
print(f'    mapping={MAPPING} w=({W_CONNECTIVITY},{W_DISTANCE}) band={DIAGONAL_BAND}')

print(f'\n  Kaydedilen dosyalar ({SAVE_DIR}):')
for f in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'    {f} ({size_kb:.0f} KB)')
print('='*80)